# 01 — SteadyAPI connection test (gol.gg)

**Goal:** Make one controlled request to a single Games of Legends game page, save the returned response, and inspect the `Gold graph & Timeline` section.

This notebook is an API and HTML-inspection test. It does **not** yet attempt to build the final event parser. We first need to see exactly what SteadyAPI returns and how the timeline is represented in the HTML.

Example page:  
`https://gol.gg/game/stats/83713/page-game/`

**Important**
- Keep your SteadyAPI token in the project-root `.env` file.
- Never paste your token directly into a notebook or commit `.env` to GitHub.
- The ScrapeFlow endpoint has a documented weight of 3 per successful request. Run the request cell only when you intend to make an API call.
- Start with `render_js=False`. If the timeline is missing from the returned HTML, we can test a second request with JavaScript rendering enabled.


## 1. Imports and project paths

This notebook expects the following project structure:

```text
golgg_scraper/
├── data/
│   └── raw/
│       ├── html/
│       └── json/
├── notebooks/
│   └── 01_api_testing.ipynb
├── .env
└── ...
```

The path-detection code below looks for the `.env` file in the project folder or its parent folders.


In [2]:
from pathlib import Path
import json
import re
import sys
import requests
import pandas as pd

from bs4 import BeautifulSoup
from dotenv import load_dotenv
import os

print("Python:", sys.version)
print("Notebook working directory:", Path.cwd())


Python: 3.14.4 (tags/v3.14.4:23116f9, Apr  7 2026, 14:10:54) [MSC v.1944 64 bit (AMD64)]
Notebook working directory: c:\Users\boyle\League_Of_Legends_Herald_Analysis\golgg_scraper\notebooks


## 2. Load the API token

Your `.env` file should contain:

```text
STEADY_API_TOKEN=your_personal_access_token_here
STEADY_API_BASE_URL=https://api.steadyapi.com
```

The token is loaded into an environment variable and is never printed.


In [3]:
# Find the project root by searching the current folder and its parents for .env
current_path = Path.cwd()
project_root = next(
    (path for path in [current_path, *current_path.parents] if (path / ".env").exists()),
    None
)

if project_root is None:
    raise FileNotFoundError(
        "Could not find .env. Place this notebook inside the project folder's "
        "'notebooks' directory, or update project_root manually."
    )

load_dotenv(project_root / ".env")

STEADY_API_TOKEN = os.getenv("STEADY_API_TOKEN")
STEADY_API_BASE_URL = os.getenv(
    "STEADY_API_BASE_URL",
    "https://api.steadyapi.com"
).rstrip("/")

if not STEADY_API_TOKEN:
    raise ValueError(
        "STEADY_API_TOKEN was not found. Check the variable name in your .env file."
    )

# Do not print the token.
print("Project root:", project_root)
print("SteadyAPI base URL:", STEADY_API_BASE_URL)
print("API token loaded:", bool(STEADY_API_TOKEN))


Project root: c:\Users\boyle\League_Of_Legends_Herald_Analysis
SteadyAPI base URL: https://api.steadyapi.com
API token loaded: True


## 3. Set the game URL and request settings

For this first test, we are requesting one game page only.

The documented ScrapeFlow endpoint is `POST /v1/scrapeflow/scraper`. The API expects the target URL in the JSON body and the personal access token in the `Authorization: Bearer ...` header.


In [4]:
GAME_ID = "83713"
GAME_URL = f"https://gol.gg/game/stats/{GAME_ID}/page-game/"

SCRAPEFLOW_ENDPOINT = f"{STEADY_API_BASE_URL}/v1/scrapeflow/scraper"

# Begin with a standard fetch. JavaScript rendering can be tested separately if needed.
RENDER_JS = False

request_headers = {
    "Authorization": f"Bearer {STEADY_API_TOKEN}",
    "Content-Type": "application/json",
    "Accept": "application/json",
}

request_payload = {
    "url": GAME_URL,
    "method": "GET",
    "geo": "US",
    "render_js": RENDER_JS,
    "timeout": 20,
    "output": "html",
}

print("Target page:", GAME_URL)
print("JavaScript rendering:", RENDER_JS)
print("Endpoint:", SCRAPEFLOW_ENDPOINT)


Target page: https://gol.gg/game/stats/83713/page-game/
JavaScript rendering: False
Endpoint: https://api.steadyapi.com/v1/scrapeflow/scraper


## 4. Make the SteadyAPI request

Run this cell once. It prints the HTTP response status and response keys, but does not expose your API token.

If the request fails, inspect the returned error message before trying again. Avoid repeatedly retrying because successful ScrapeFlow requests consume API allowance.


In [5]:
response = requests.post(
    SCRAPEFLOW_ENDPOINT,
    headers=request_headers,
    json=request_payload,
    timeout=45,
)

print("HTTP status:", response.status_code)
response.raise_for_status()

response_data = response.json()

print("Top-level response keys:", list(response_data.keys()))
print("API success field:", response_data.get("success", "Not supplied"))

meta = response_data.get("meta", {})
print("Target page status (meta.status):", meta.get("status"))
print("Target page status_code:", meta.get("status_code"))
print("Response message:", response_data.get("message", "No message returned"))


HTTP status: 200
Top-level response keys: ['meta', 'body']
API success field: Not supplied
Target page status (meta.status): 200
Target page status_code: None
Response message: No message returned


## 5. Save the raw API response

We save both the complete JSON response and the HTML body (if the API returned HTML as a string).

Keeping the original response is useful when we later refine the parser or need to investigate a missing event.


In [6]:
raw_json_dir = project_root / "data" / "raw" / "json"
raw_html_dir = project_root / "data" / "raw" / "html"

raw_json_dir.mkdir(parents=True, exist_ok=True)
raw_html_dir.mkdir(parents=True, exist_ok=True)

json_path = raw_json_dir / f"game_{GAME_ID}_steadyapi_response.json"

with json_path.open("w", encoding="utf-8") as file:
    json.dump(response_data, file, ensure_ascii=False, indent=2)

# ScrapeFlow documents the fetched page content in the response's "body" field.
html_content = response_data.get("body")

if isinstance(html_content, dict):
    print("The response body is a JSON object, not an HTML string.")
    print("Body keys:", list(html_content.keys()))
elif isinstance(html_content, str):
    html_path = raw_html_dir / f"game_{GAME_ID}.html"
    html_path.write_text(html_content, encoding="utf-8")
    print("HTML saved:", html_path)
    print("HTML characters:", len(html_content))
else:
    print("No HTML string found in response_data['body'].")
    print("Inspect response_data to determine the returned response format.")

print("Raw JSON saved:", json_path)


HTML saved: c:\Users\boyle\League_Of_Legends_Herald_Analysis\data\raw\html\game_83713.html
HTML characters: 625498
Raw JSON saved: c:\Users\boyle\League_Of_Legends_Herald_Analysis\data\raw\json\game_83713_steadyapi_response.json


## 6. Check whether the timeline is present in the returned HTML

We are specifically looking for the `Gold graph & Timeline` heading and nearby text.

This is a diagnostic check, not the final event extraction. The page may represent events using images, icons, CSS classes, or other markup that we need to inspect before writing a reliable parser.


In [7]:
if not isinstance(html_content, str):
    raise TypeError(
        "The API did not return HTML as a string. Review the response body above."
    )

soup = BeautifulSoup(html_content, "lxml")
page_text = soup.get_text(" ", strip=True)

timeline_present = "Gold graph & Timeline" in page_text

print("HTML contains 'Gold graph & Timeline':", timeline_present)
print("HTML title:", soup.title.get_text(strip=True) if soup.title else "No title")
print("Page text length:", len(page_text))

if timeline_present:
    heading_match = re.search(r"Gold graph\s*&\s*Timeline", page_text, flags=re.I)
    start = max(0, heading_match.start() - 200)
    end = min(len(page_text), heading_match.end() + 1800)
    print("\nText near timeline heading:\n")
    print(page_text[start:end])
else:
    print("\nTimeline heading was not found in the returned HTML.")
    print("This may mean JavaScript rendering is required, or the page markup differs.")


HTML contains 'Gold graph & Timeline': True
HTML title: IG vs Gen.G game 1 - Worlds 2026 Main Event SEMIFINALS - Games of Legends
Page text length: 4318

Text near timeline heading:

eld Bash Total Damage: 232 Ultimate Hunter Total Stacks: 4 Bone Plating Total Damage Blocked: 868 +8 Ability Haste Move Speed Health Unflinching Seconds in combat with bonus resistances: 134 0/4/4 37 Gold graph & Timeline GEN IG 3:45 5:43 11:20 14:30 15:27 16:47 22:38 28:30 30:10 Plates GEN IG Voidgrubs 1 2 Plates 37 18 Plates TOP 15 6 Plates MID 15 7 Plates BOT 7 5 Gold distribution GEN IG TOP 19.3% 16.8% JUNGLE 25.3% 23.2% MID 17.6% 19.8% ADC 23.5% 26.7% SUPPORT 14.3% 13.6% Damage distribution GEN IG TOP 22.5% 22.9% JUNGLE 28.8% 11.3% MID 13.7% 27% ADC 29.4% 33.4% SUPPORT 5.5% 5.4% Vision Jungle share CS killed in team jungle + CS killed in enemy jungle © 2026 Games Of Legends - Private Policy Games Of Legends isn’t endorsed by Riot Games and doesn’t reflect the views or opinions of Riot Games or anyone 

## 7. Inspect the timeline's HTML structure

This cell locates the timeline heading and prints its parent elements. We want to understand how the two team lanes, event icons, and timestamps are grouped in the DOM.

The output is intentionally limited to avoid dumping the entire page.


In [8]:
timeline_heading = soup.find(
    lambda tag: tag.name in ["h1", "h2", "h3", "h4", "div", "span"]
    and tag.get_text(" ", strip=True).lower() == "gold graph & timeline"
)

if timeline_heading is None:
    # Fallback: find any element whose visible text contains the heading.
    timeline_heading = soup.find(
        lambda tag: tag.name and tag.get_text(" ", strip=True).lower() == "gold graph & timeline"
    )

if timeline_heading is not None:
    print("Heading element:", timeline_heading.name)
    print("Heading classes:", timeline_heading.get("class"))

    ancestor = timeline_heading
    for level in range(1, 5):
        if ancestor is None:
            break
        print(f"\n--- Parent level {level - 1}: <{ancestor.name}> ---")
        print("Classes:", ancestor.get("class"))
        print(ancestor.prettify()[:6000])
        ancestor = ancestor.parent
else:
    print("Could not isolate a timeline heading element with BeautifulSoup.")
    print("Use the text output above and inspect the saved HTML file in VS Code.")


Heading element: tr
Heading classes: None

--- Parent level 0: <tr> ---
Classes: None
<tr>
 <th>
  Gold graph &amp; Timeline
 </th>
</tr>


--- Parent level 1: <table> ---
Classes: ['table_list']
<table class="table_list" style="height:140px">
 <tr>
  <th>
   Gold graph &amp; Timeline
  </th>
 </tr>
 <tr>
  <td>
   <div class="row m-2">
    <div class="col-12">
     <div style="width:100%;padding:15px">
      <canvas id="GoldLine" style="max-height:120px;max-width:1280px">
      </canvas>
     </div>
    </div>
   </div>
   <div class="row">
    <div class="col-12 flex-wrap">
     <span style="display:inline-block">
      <span class="empty_action title_blue">
       GEN
       <br/>
       <br/>
      </span>
      <span class="empty_action title_red">
       IG
       <br/>
       <br/>
      </span>
     </span>
     <span style="display:inline-block">
      <span class="blue_action">
       <img alt="First blood" class="champion_icon_light" src="../_img/firstblood3.png"/>
       <b

## 8. Inspect image labels and timestamps near the timeline

On gol.gg, event icons may be images whose `alt` attributes describe the event (for example, First blood, Rift Herald, or Nashor). Timestamps may be separate text nodes.

This cell reports image alt text and timestamp-looking strings from the timeline container if we found one. It is exploratory; it does not yet assign events to teams.


In [9]:
if timeline_heading is not None:
    # Use a parent container for a focused inspection.
    timeline_container = timeline_heading.parent
    for _ in range(2):
        if timeline_container and timeline_container.parent:
            timeline_container = timeline_container.parent

    timeline_images = [
        {
            "alt": img.get("alt"),
            "src": img.get("src"),
            "class": img.get("class"),
        }
        for img in timeline_container.find_all("img")
    ]

    timeline_text = timeline_container.get_text(" ", strip=True)
    timestamps = re.findall(r"(?<!\d)\d{1,2}:\d{2}(?!\d)", timeline_text)

    print("Image count in inspected container:", len(timeline_images))
    print("\nImage labels:")
    for item in timeline_images:
        print(item)

    print("\nTimestamp-looking strings:")
    print(timestamps)

    print("\nTimeline container text:")
    print(timeline_text[:5000])
else:
    print("Timeline heading not found; cannot inspect its container yet.")


Image count in inspected container: 9

Image labels:
{'alt': 'First blood', 'src': '../_img/firstblood3.png', 'class': ['champion_icon_light']}
{'alt': 'Chemtech Drake', 'src': '../_img/chemtech-dragon.png', 'class': ['champion_icon_light']}
{'alt': 'Mountain Drake', 'src': '../_img/mountain-dragon.png', 'class': ['champion_icon_light']}
{'alt': 'First Tower', 'src': '../_img/firsttower2.png', 'class': ['champion_icon_light']}
{'alt': 'Rift Herald', 'src': '../_img/herald-icon.png', 'class': ['champion_icon_light']}
{'alt': 'Hextech Drake', 'src': '../_img/hextech-dragon.png', 'class': ['champion_icon_light']}
{'alt': 'Hextech Drake', 'src': '../_img/hextech-dragon.png', 'class': ['champion_icon_light']}
{'alt': 'Hextech Drake', 'src': '../_img/hextech-dragon.png', 'class': ['champion_icon_light']}
{'alt': 'Nashor', 'src': '../_img/nashor-icon.png', 'class': ['champion_icon_light']}

Timestamp-looking strings:
['3:45', '5:43', '11:20', '14:30', '15:27', '16:47', '22:38', '28:30', '30:1

In [10]:

# Inspect the HTML hierarchy for each timeline event

timeline_imgs = timeline_container.find_all("img")

for i, img in enumerate(timeline_imgs):

    print("\n" + "=" * 70)
    print(f"EVENT ICON {i + 1}: {img.get('alt')}")

    # Print the image itself
    print("\nImage:")
    print(img)

    # Inspect the image's parent and grandparents
    parent = img.parent

    for level in range(1, 4):

        if parent is None:
            break

        print(f"\nParent level {level}:")
        print("Tag:", parent.name)
        print("Classes:", parent.get("class"))
        print("ID:", parent.get("id"))

        # Print a limited amount of HTML
        print(parent.prettify()[:2500])

        parent = parent.parent



EVENT ICON 1: First blood

Image:
<img alt="First blood" class="champion_icon_light" src="../_img/firstblood3.png"/>

Parent level 1:
Tag: span
Classes: ['blue_action']
ID: None
<span class="blue_action">
 <img alt="First blood" class="champion_icon_light" src="../_img/firstblood3.png"/>
 <br/>
 3:45
</span>


Parent level 2:
Tag: span
Classes: None
ID: None
<span style="display:inline-block">
 <span class="blue_action">
  <img alt="First blood" class="champion_icon_light" src="../_img/firstblood3.png"/>
  <br/>
  3:45
 </span>
 <span class="empty_action">
  <br/>
 </span>
</span>


Parent level 3:
Tag: div
Classes: ['col-12', 'flex-wrap']
ID: None
<div class="col-12 flex-wrap">
 <span style="display:inline-block">
  <span class="empty_action title_blue">
   GEN
   <br/>
   <br/>
  </span>
  <span class="empty_action title_red">
   IG
   <br/>
   <br/>
  </span>
 </span>
 <span style="display:inline-block">
  <span class="blue_action">
   <img alt="First blood" class="champion_icon_li

## 9. Initial result checklist

After running the notebook, check the following:

- [ ] The API token loaded successfully.
- [ ] SteadyAPI returned HTTP 200.
- [ ] `meta.status` indicates the target page returned successfully.
- [ ] The response body contains HTML.
- [ ] The HTML contains `Gold graph & Timeline`.
- [ ] The saved HTML contains event labels and timestamps.
- [ ] The HTML structure makes it possible to distinguish the GEN and IG timeline lanes.

**Do not assume that every timestamp belongs to the nearest event or that every event belongs to a particular team yet.** We will inspect the actual markup and build team attribution rules in the next notebook.

### If the timeline is missing

Only if the standard fetch does not include the timeline, change:

```python
RENDER_JS = True
```

Then rerun the request and save cells. This makes another API request and may take longer. The ScrapeFlow documentation also supports `wait_for_selector`, which we can use once we identify the timeline's CSS selector.
